# Agent 5/6 â€” ESG Scoring & Climate

**What this notebook does:**  
Takes the raw ESG numbers from the provided datasets and converts them into clean, comparable scores (0â€“100) for E (Environmental), S (Social), G (Governance), and a combined ESG score. Pillar weights are sector-specific, calibrated to SASB materiality standards.

**Why we don't just use one vendor ESG score:**  
Academic research (Berg, Koelbel & Rigobon 2022 â€” *Aggregate Confusion*) shows that ESG scores from different vendors disagree significantly. So we build our own transparent score from raw variables â€” this way we can explain every number to the investor panel.

**Why SASB materiality?**  
Applying the same E/S/G weights to every company ignores sector context. SASB identifies which ESG factors are financially material per industry â€” we use these to set pillar weights before scoring.

**How to present this to investors:**  
> *Rather than relying on a black-box third-party ESG rating, we constructed our own transparent score from raw environmental, social, and governance variables. Pillar weights are calibrated to SASB materiality standards so that, for example, an energy company is judged primarily on its environmental performance while a bank is judged primarily on governance quality.*

In [1]:
import pandas as pd
import numpy as np
from datetime import date
import glob

# Load master dataset from notebook 01
master_files = sorted(glob.glob("../outputs/scores/master_dataset_*.csv"))
if not master_files:
    raise FileNotFoundError("Run notebook 01 first to create the master dataset.")

master = pd.read_csv(master_files[-1])
print(f"Master dataset loaded: {len(master)} companies")
print("Columns available:", list(master.columns))

Master dataset loaded: 167 companies
Columns available: ['rc', 'idBbGlobal', 'idBbUnique', 'idBbCompany', 'idBbGlobalCompany', 'idBbGlobalCompanyName', 'idBbSecNumDes', 'ticker', 'exchCode', 'idIsin', 'idCusip', 'idSedol1', 'securityTyp', 'securityTyp2', 'classificationLevelCode1', 'classificationLevelName1', 'classificationLevelCode2', 'classificationLevelName2', 'classificationLevelCode3', 'classificationLevelName3', 'classificationLevelCode4', 'classificationLevelName4', 'classificationLevelCode5', 'classificationLevelName5', 'classificationLevelCode6', 'classificationLevelName6', 'classificationLevelCode7', 'classificationLevelName7', 'classificationScheme', 'rank', 'company', 'return_5y_pct', 'return_10y_pct', 'bb_ticker', 'yf_ticker', 'excluded', 'latestPeriodEndCsr', 'esgDisclosureScore', 'socialDisclosureScore', 'environDisclosureScore', 'directCo2Emissions', 'indirectCo2Emissions', 'totalCo2Emissions', 'totalGhgEmissions', 'scp3BusinessTravelEmissions', 'gasFlaring', 'emission

## Step 0 â€” SASB Materiality Weights

The Sustainability Accounting Standards Board (SASB) identifies which ESG factors are **financially material** for each industry sector. Rather than applying the same E/S/G pillar weights to every company, we adjust the weights based on the sector each company operates in.

**Why this matters:** A bank and an oil company should not be scored on the same ESG priorities. For an oil company, the Environmental pillar dominates. For a bank, Governance (data security, systemic risk, executive pay) is most material.

| Sector | E weight | S weight | G weight | Key SASB rationale |
|--------|----------|----------|----------|--------------------|
| Technology | 25% | 35% | 40% | Data security, employee engagement, governance |
| Financials | 20% | 35% | 45% | Systemic risk, data privacy, governance quality |
| Energy | 55% | 25% | 20% | GHG emissions, water, spills dominate |
| Consumer Discretionary | 35% | 40% | 25% | Supply chain labour, product safety |
| Consumer Staples | 35% | 40% | 25% | Supply chain, packaging, labour practices |
| Health Care | 25% | 40% | 35% | Patient safety, access, product quality |
| Industrials | 45% | 35% | 20% | GHG, occupational health, waste |
| Materials | 50% | 30% | 20% | Air/water pollution, hazardous waste |
| Real Estate | 45% | 25% | 30% | Energy efficiency, water management |
| Communication Services | 20% | 40% | 40% | Data privacy, content governance, diversity |
| Utilities | 50% | 25% | 25% | GHG intensity, renewable mix, water |
| Default (unmatched) | 40% | 30% | 30% | Balanced fallback |


In [2]:
# Create bics_sector alias from the Bloomberg classification column
if "bics_sector" not in master.columns:
    master["bics_sector"] = master["classificationLevelName1"]

# SASB materiality weights per BICS sector: (E_weight, S_weight, G_weight)
SASB_WEIGHTS = {
    "Technology":              (0.25, 0.35, 0.40),
    "Financials":              (0.20, 0.35, 0.45),
    "Energy":                  (0.55, 0.25, 0.20),
    "Consumer Discretionary":  (0.35, 0.40, 0.25),
    "Consumer Staples":        (0.35, 0.40, 0.25),
    "Health Care":             (0.25, 0.40, 0.35),
    "Industrials":             (0.45, 0.35, 0.20),
    "Materials":               (0.50, 0.30, 0.20),
    "Real Estate":             (0.45, 0.25, 0.30),
    "Communication Services":  (0.20, 0.40, 0.40),
    "Utilities":               (0.50, 0.25, 0.25),
}
DEFAULT_WEIGHTS = (0.40, 0.30, 0.30)

# Map each company to its sector weights
def get_sasb_weights(sector):
    for key in SASB_WEIGHTS:
        if isinstance(sector, str) and key.lower() in sector.lower():
            return SASB_WEIGHTS[key]
    return DEFAULT_WEIGHTS

master["sasb_e_weight"], master["sasb_s_weight"], master["sasb_g_weight"] = zip(
    *master["bics_sector"].apply(get_sasb_weights)
)

# Show the weight assigned to each sector present in the universe
weight_summary = (
    master[["bics_sector", "sasb_e_weight", "sasb_s_weight", "sasb_g_weight"]]
    .drop_duplicates("bics_sector")
    .sort_values("bics_sector")
    .reset_index(drop=True)
)
print(f"Sectors in universe: {len(weight_summary)}")
print(weight_summary.to_string(index=False))

Sectors in universe: 10
           bics_sector  sasb_e_weight  sasb_s_weight  sasb_g_weight
        Communications           0.40           0.30           0.30
Consumer Discretionary           0.35           0.40           0.25
      Consumer Staples           0.35           0.40           0.25
                Energy           0.55           0.25           0.20
            Financials           0.20           0.35           0.45
           Health Care           0.25           0.40           0.35
           Industrials           0.45           0.35           0.20
             Materials           0.50           0.30           0.20
            Technology           0.25           0.35           0.40
             Utilities           0.50           0.25           0.25


## Step 1 â€” Choose which variables to include

After running the cell above, look at the column list and decide which columns belong to E, S, and G.

**Examples of typical columns:**
- **E (Environmental):** GHG emissions, Scope 1, Scope 2, carbon intensity, water usage, energy consumption
- **S (Social):** employee turnover, workplace accidents, gender pay gap, employee training hours
- **G (Governance):** board independence, board gender diversity, CEO pay ratio, audit committee independence

In [3]:
E_VARS = [
    "scope1_emissions_tco2e",
    "carbon_intensity_tco2e_per_eur_m_revenue",
    "renewable_energy_pct",
    "water_withdrawal_m3",
    "waste_recycled_pct",
]

S_VARS = [
    "women_in_workforce_pct",
    "gender_pay_gap_pct",
    "employee_turnover_pct",
    "work_related_injury_rate",
    "training_hours_per_employee",
]

G_VARS = [
    "board_independence_pct",
    "women_on_board_pct",
    "audit_committee_independence_pct",
    "executive_pay_esg_linked_pct",
    "ceo_pay_ratio",
]

# True = lower is better (e.g. emissions), False = higher is better
INVERT = {
    "scope1_emissions_tco2e":                  True,
    "carbon_intensity_tco2e_per_eur_m_revenue": True,
    "renewable_energy_pct":                    False,
    "water_withdrawal_m3":                     True,
    "waste_recycled_pct":                      False,
    "women_in_workforce_pct":                  False,
    "gender_pay_gap_pct":                      True,
    "employee_turnover_pct":                   True,
    "work_related_injury_rate":                True,
    "training_hours_per_employee":             False,
    "board_independence_pct":                  False,
    "women_on_board_pct":                      False,
    "audit_committee_independence_pct":        False,
    "executive_pay_esg_linked_pct":            False,
    "ceo_pay_ratio":                           True,
}

ALL_VARS = E_VARS + S_VARS + G_VARS
print(f"E variables: {len(E_VARS)}")
print(f"S variables: {len(S_VARS)}")
print(f"G variables: {len(G_VARS)}")

E variables: 5
S variables: 5
G variables: 5


## Step 2 â€” Normalise all variables to 0â€“100

We use min-max normalisation: the best company in each variable scores 100, the worst scores 0, everyone else is in between. This makes all variables comparable regardless of their units.

In [4]:
def normalise(series, invert=False):
    """Scale a series to 0-100. If invert=True, lower values get higher scores."""
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(50.0, index=series.index)  # all same value â†’ 50
    scaled = (series - mn) / (mx - mn) * 100
    return 100 - scaled if invert else scaled

scores = master[["ticker"] if "ticker" in master.columns else [master.columns[0]]].copy()

# Normalise each variable
for col in ALL_VARS:
    if col in master.columns:
        inv = INVERT.get(col, False)
        scores[f"{col}_score"] = normalise(master[col], invert=inv)
    else:
        print(f"WARNING: Column '{col}' not found in master dataset")

print("Variables normalised.")
scores.head(3)

Variables normalised.


,ticker,scope1_emissions_tco2e_score,carbon_intensity_tco2e_per_eur_m_revenue_score,renewable_energy_pct_score,water_withdrawal_m3_score,waste_recycled_pct_score,women_in_workforce_pct_score,gender_pay_gap_pct_score,employee_turnover_pct_score,work_related_injury_rate_score,training_hours_per_employee_score,board_independence_pct_score,women_on_board_pct_score,audit_committee_independence_pct_score,executive_pay_esg_linked_pct_score,ceo_pay_ratio_score
0,SEBA,100.000000,NaN,0.172435,NaN,57.00,59.677214,NaN,76.509279,NaN,8.176215,37.062550,81.481481,39.999760,0.0,74.940101
1,ALV,99.978110,NaN,1.229094,99.983898,46.60,54.593976,99.999787,66.408269,96.074271,12.337591,42.307914,58.518519,51.999808,100.0,90.759830
2,E7V,99.950919,NaN,NaN,NaN,3.64,17.867582,NaN,65.703547,66.185676,3.930188,61.538994,NaN,100.000000,100.0,96.140671


## Step 3 â€” Calculate E, S, G and combined ESG scores

We weight each pillar equally (33%/33%/33%) unless you want to change this.  
The combined ESG score is a weighted average of E, S, and G.

In [5]:
# Per-company SASB weights are already in master["sasb_e_weight"] etc.
# We merge them into the scores dataframe and apply row-by-row.

def pillar_score(df, variables):
    cols = [f"{v}_score" for v in variables if f"{v}_score" in df.columns]
    if not cols:
        return pd.Series(np.nan, index=df.index)
    return df[cols].mean(axis=1)

scores["E_score"] = pillar_score(scores, E_VARS).round(2)
scores["S_score"] = pillar_score(scores, S_VARS).round(2)
scores["G_score"] = pillar_score(scores, G_VARS).round(2)

# Merge SASB weights into scores
scores = scores.merge(
    master[["ticker", "bics_sector", "sasb_e_weight", "sasb_s_weight", "sasb_g_weight"]],
    on="ticker", how="left"
)

# Apply sector-specific weights row by row
scores["ESG_score"] = (
    scores["E_score"].fillna(50) * scores["sasb_e_weight"] +
    scores["S_score"].fillna(50) * scores["sasb_s_weight"] +
    scores["G_score"].fillna(50) * scores["sasb_g_weight"]
).round(2)

final_scores = scores[[
    "ticker", "bics_sector",
    "E_score", "S_score", "G_score", "ESG_score",
    "sasb_e_weight", "sasb_s_weight", "sasb_g_weight"
]]

print("Top 10 by ESG score (SASB-weighted):")
print(final_scores.nlargest(10, "ESG_score")[
    ["ticker", "bics_sector", "E_score", "S_score", "G_score",
     "sasb_e_weight", "sasb_s_weight", "sasb_g_weight", "ESG_score"]
].to_string(index=False))

Top 10 by ESG score (SASB-weighted):
ticker            bics_sector  E_score  S_score    G_score  sasb_e_weight  sasb_s_weight  sasb_g_weight  ESG_score
   NXG Consumer Discretionary    98.99    76.66  86.480173           0.35           0.40           0.25      86.93
  IHCB                 Energy   100.00    49.23  94.550787           0.55           0.25           0.20      86.22
  D9F2              Utilities    99.23    51.67  87.991493           0.50           0.25           0.25      84.53
  SKWA              Materials    93.32    60.28  74.142948           0.50           0.30           0.20      79.57
  TW11              Materials    98.95    61.69  50.550392           0.50           0.30           0.20      78.09
   B3K            Industrials    99.35    67.14  44.259443           0.45           0.35           0.20      77.06
   NOT            Health Care    70.00    68.21  91.935829           0.25           0.40           0.35      76.96
   UEI              Utilities    99.97    5

C:\Users\ionva\AppData\Local\Temp\ipykernel_2564\2541061102.py:24: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  scores["G_score"].fillna(50) * scores["sasb_g_weight"]


## Step 4 â€” ESG Triangulation

We cross-check our own ESG score against two external sources:
- **Bloomberg ESG Disclosure Score** â€” from the professor's provided CSV. Measures how completely a company discloses ESG data. Range 0â€“100, higher is better. Threshold: â‰¥ 50 = PASS.
- **Sustainalytics Risk Score** â€” fetched via yfinance (Yahoo Finance ESG tab). Measures unmanaged ESG risk. Range 0â€“100, **lower is better**. Threshold: â‰¤ 30 (medium risk or below) = PASS.

**Rule:** a company must pass at least **2 of 3** sources to clear triangulation. Companies that pass only 1 or fewer are flagged as **WATCHLIST** â€” they remain in the universe but are flagged for manual review before final selection.

| Source | Pass condition | Rationale |
|--------|---------------|----------|
| Agent score | â‰¥ 50 / 100 | Above-median in our own scoring |
| Bloomberg ESG Disclosure | â‰¥ 50 / 100 | Sufficient ESG transparency |
| Sustainalytics Risk | â‰¤ 30 / 100 | Medium risk or below |

In [6]:
import yfinance as yf
import time

# Source 2: Bloomberg ESG Disclosure Score
# Use idBbCompany (unique per company) to avoid ticker ambiguity
target_ids = set(master["idBbCompany"].dropna().astype(int).tolist())
esg_raw = pd.read_csv("../data/provided/esgEnvironmentalSocialConsolidatedV4.csv",
                      usecols=["idBbCompany", "esgDisclosureScore"])
esg_raw = esg_raw[esg_raw["idBbCompany"].isin(target_ids)].dropna(subset=["esgDisclosureScore"])
esg_raw = esg_raw.drop_duplicates("idBbCompany", keep="last")

id_to_ticker = master[["idBbCompany", "ticker"]].dropna().drop_duplicates("idBbCompany")
bloomberg_esg = id_to_ticker.merge(esg_raw, on="idBbCompany", how="left")
bloomberg_esg = bloomberg_esg[["ticker", "esgDisclosureScore"]].rename(
    columns={"esgDisclosureScore": "bloomberg_esg_disclosure"}
)
n_matched = bloomberg_esg["bloomberg_esg_disclosure"].notna().sum()
print(f"Bloomberg ESG disclosure matched: {n_matched} / {len(bloomberg_esg)} companies")

# Source 3: Sustainalytics via yfinance (use yf_ticker, not Bloomberg ticker)
yf_map = master[["ticker", "yf_ticker"]].dropna(subset=["yf_ticker"]).drop_duplicates("ticker")
print(f"Fetching Sustainalytics for {len(yf_map)} tickers...")

sust_rows = []
for i, row in enumerate(yf_map.itertuples(), 1):
    try:
        sust = yf.Ticker(row.yf_ticker).sustainability
        score = sust.loc["totalEsg", "Value"] if (sust is not None and "totalEsg" in sust.index) else None
    except Exception:
        score = None
    sust_rows.append({"ticker": row.ticker, "sustainalytics_risk_score": score})
    if i % 10 == 0:
        print(f"  {i}/{len(yf_map)} done...")
        time.sleep(1)

sust_df = pd.DataFrame(sust_rows)
n_sust = sust_df["sustainalytics_risk_score"].notna().sum()
print(f"Sustainalytics fetched: {n_sust} / {len(sust_df)} companies")

# Merge all three sources (all joins on Bloomberg ticker â€” 1:1 with final_scores)
tri = (
    final_scores[["ticker", "ESG_score"]]
    .merge(bloomberg_esg, on="ticker", how="left")
    .merge(sust_df,       on="ticker", how="left")
)

AGENT_THRESHOLD          = 50
BLOOMBERG_THRESHOLD      = 50
SUSTAINALYTICS_THRESHOLD = 30

tri["agent_pass"]          = tri["ESG_score"] >= AGENT_THRESHOLD
tri["bloomberg_pass"]      = tri["bloomberg_esg_disclosure"] >= BLOOMBERG_THRESHOLD
tri["sustainalytics_pass"] = tri["sustainalytics_risk_score"] <= SUSTAINALYTICS_THRESHOLD

tri["tri_sources_available"] = (
    tri["ESG_score"].notna().astype(int) +
    tri["bloomberg_esg_disclosure"].notna().astype(int) +
    tri["sustainalytics_risk_score"].notna().astype(int)
)
tri["tri_passes"] = (
    tri["agent_pass"].fillna(False).astype(int) +
    tri["bloomberg_pass"].fillna(False).astype(int) +
    tri["sustainalytics_pass"].fillna(False).astype(int)
)

def triangulation_result(row):
    if row["tri_sources_available"] < 2:
        return "LOW_DATA"
    return "PASS" if row["tri_passes"] >= 2 else "WATCHLIST"

tri["triangulation_result"] = tri.apply(triangulation_result, axis=1)

final_scores = final_scores.merge(
    tri[["ticker", "bloomberg_esg_disclosure", "sustainalytics_risk_score",
         "tri_sources_available", "tri_passes", "triangulation_result"]],
    on="ticker", how="left"
)

print("Triangulation results:")
print(final_scores["triangulation_result"].value_counts().to_string())
print()
watchlist = final_scores[final_scores["triangulation_result"] == "WATCHLIST"]
print(f"WATCHLIST ({len(watchlist)} companies):")
print(watchlist[["ticker", "ESG_score", "bloomberg_esg_disclosure",
                  "sustainalytics_risk_score", "tri_passes"]].to_string(index=False))


Bloomberg ESG disclosure matched: 162 / 167 companies
Fetching Sustainalytics for 167 tickers...


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"No fundamentals data found for symbol: SEB-A.ST"}}}


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"No fundamentals data found for symbol: ALV.DE"}}}


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"No fundamentals data found for symbol: RXL.PA"}}}


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"No fundamentals data found for symbol: ISP.MI"}}}


...

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"No fundamentals data found for symbol: DNB.OL"}}}


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"No fundamentals data found for symbol: RILBA.CO"}}}


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"No fundamentals data found for symbol: SB1NO.OL"}}}


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"No fundamentals data found for symbol: SBNOR.OL"}}}


Sustainalytics fetched: 0 / 167 companies
Triangulation results:
triangulation_result
PASS         93
WATCHLIST    69
LOW_DATA      5

WATCHLIST (69 companies):
ticker  ESG_score  bloomberg_esg_disclosure sustainalytics_risk_score  tri_passes
  SEBA      48.33                     59.42                      None           1
   RWE      47.23                     64.13                      None           1
   GBF      52.49                     40.84                      None           1
   HOT      47.92                     59.23                      None           1
  TCO0      62.94                     48.12                      None           1
   BSI      50.21                     44.28                      None           1
   8RJ      42.27                     50.84                      None           1
  SUL1      68.78                     46.10                      None           1
   IVS      59.19                     38.26                      None           1
  NCYD      61.60  

## Step 5 â€” WACI (Weighted Average Carbon Intensity)

WACI is the standard carbon metric required by the assignment.  
It measures how carbon-intensive the portfolio is, weighted by how much we invest in each company.  
Unit: tonnes of COâ‚‚ per million euros of revenue (tCOâ‚‚e / â‚¬m revenue)

In [7]:
# Carbon intensity: use Bloomberg Calc value where available,
# fall back to sector median for missing companies.
# Bloomberg co2IntensityPerSalesCalc is an estimated/modelled field.
# Imputed values are flagged ci_source = sector_median_imputed.

CARBON_COL   = "carbon_intensity_tco2e_per_eur_m_revenue"
SECTOR_COL_W = "classificationLevelName1"

ci = master[["ticker", CARBON_COL, SECTOR_COL_W]].drop_duplicates("ticker").copy()
ci.columns = ["ticker", "carbon_intensity", "sector"]

# Compute sector medians from reported values
sector_medians = ci.dropna(subset=["carbon_intensity"]).groupby("sector")["carbon_intensity"].median()
global_median  = ci["carbon_intensity"].median()

print("Sector median carbon intensity (tCO2e/EUR m revenue):")
for sector, median in sector_medians.sort_values(ascending=False).items():
    n_reported = ci[ci["sector"] == sector]["carbon_intensity"].notna().sum()
    n_total    = (ci["sector"] == sector).sum()
    print(f"  {sector:<30} {median:>8.1f}  ({n_reported}/{n_total} reported)")

# Impute missing values with sector median (or global if sector has no data)
def impute_ci(row):
    if pd.notna(row["carbon_intensity"]):
        return row["carbon_intensity"], "bloomberg_calc"
    fallback = sector_medians.get(row["sector"], global_median)
    return fallback, "sector_median_imputed"

ci[["carbon_intensity", "ci_source"]] = ci.apply(
    lambda r: pd.Series(impute_ci(r)), axis=1
)

n_reported = (ci["ci_source"] == "bloomberg_calc").sum()
n_imputed  = (ci["ci_source"] == "sector_median_imputed").sum()
print(f"\nCoverage: {n_reported} reported, {n_imputed} sector-median imputed")
print(f"\nTop 10 highest carbon intensity:")
print(ci.sort_values("carbon_intensity", ascending=False)[["ticker","carbon_intensity","ci_source"]].head(10).to_string(index=False))


Carbon intensity (tCO2e per EUR m revenue) â€” top 10 lowest (cleanest):
ticker  carbon_intensity
   2FE              0.00
   HOT              0.00
  HNR1              0.09
   2NN              0.26
   TM2              0.35
  BAKA              0.72
  MUV2              1.36
  SVKB              4.87
   P9O              5.80
   UNC              7.99

Top 10 highest (most carbon-intensive):
ticker  carbon_intensity
   UCM           6439.03
  HLBN           4285.61
   HEI           3982.39
   EAM            629.82
  PKY1            134.38
   C0Q             85.39
   SOC             53.27
   OFK             23.48
  TLLB             14.68
   UNC              7.99


## Step 6 â€” Save ESG scores

In [8]:
today = str(date.today())

# Merge carbon intensity + imputation source into ESG scores
if "carbon_intensity" in ci.columns:
    final_scores = final_scores.merge(
        ci[["ticker", "carbon_intensity", "ci_source"]], on="ticker", how="left"
    )

final_scores["data_vintage"] = today
out_path = f"../outputs/scores/esg_scores_{today}.csv"
final_scores.to_csv(out_path, index=False)
print(f"ESG scores saved: {out_path}")
print(f"Columns: {list(final_scores.columns)}")
print("\nScore summary:")
final_scores[["E_score","S_score","G_score","ESG_score"]].describe().round(2)


ESG scores saved: ../outputs/scores/esg_scores_2026-05-13.csv
Score summary:


,E_score,S_score,ESG_score
count,152.00,155.00,167.00
mean,72.31,51.29,62.23
std,20.45,16.91,9.81
min,0.00,0.00,37.71
25%,55.19,40.34,55.34
50%,69.77,51.75,62.68
75%,94.49,62.40,68.82
max,100.00,86.06,86.93


## Notebook complete

You now have:
- **SASB materiality weights** per sector
- **E, S, G and combined ESG scores** (0â€“100), sector-weighted
- **ESG Triangulation** â€” Bloomberg disclosure + Sustainalytics cross-check, 2-of-3 rule
- **WATCHLIST flags** for companies that fail triangulation
- **Carbon intensity** per company

Output files:
- `outputs/scores/esg_scores_DATE.csv` â€” full ESG scores with triangulation columns

**Next:** Open `agent11_portfolio_construction.ipynb` to select the final 15â€“25 holdings.